In [ ]:
# YENİ HÜCRE 0: Google Drive'a Bağlan
from google.colab import drive
drive.mount('/content/drive')

# Çalışma klasörünü kontrol et
import os
EN2TR_DIR = '/content/drive/MyDrive/Tez_Veri/EN2TR'
os.makedirs(EN2TR_DIR, exist_ok=True)
print(f"✅ Drive bağlandı. Çalışma klasörü: {EN2TR_DIR}")
print(f"   Klasör içeriği: {os.listdir(EN2TR_DIR)}")

Mounted at /content/drive
✅ Drive bağlandı. Çalışma klasörü: /content/drive/MyDrive/Tez_Veri/EN2TR
   Klasör içeriği: ['TR2EN.txt', 'train.tsv', 'processed_train_graph_data.pt', 'processed_valid_graph_data.pt', 'processed_test_graph_data.pt', 'val.tsv', 'test.tsv', 'bpe_tokenizer_en2tr.json', 'morph_vocab_en2tr.pkl']


In [ ]:
# HÜCRE 1: Bağımlılığı Düzelt
!pip uninstall -y antlr4-python3-runtime
!pip install antlr4-python3-runtime==4.9.3

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=5141cfd44d6ed10da236d3a64dda5cbad706179560a1052ef834abc983b46084
  Stored in directory: /root/.cache/pip/wheels/d5/b3/74/a35b66048c9de6631cd74cbc9475e6feb3e69a467983446bd8
Successfully built antlr4-python3-runtime


In [ ]:
#HÜCRE 3
import torch
!pip install zemberek-python
!pip install jpype1
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-geometric
print("\nKurulumlar tamamlandı. Zemberek'in Java'yı bulabilmesi için Runtime'ı yeniden başlatmanız gerekebilir.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 MB 110.0 MB/s  0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.8-py3-none-any.whl size=141246 sha256=9ae5993b1ef8235f55fb2ac36203396e034a89fcdd1047910d67a56b1154ae31
  Stored in directory: /root/.cache/pip/wheels/8a/7b/27/7537e1a791b9e39a179379f3fb4529e2a480a98411c1b9ebf4
Successfully built antlr4-python3-runtime
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.9.3
    Uninstalling antlr4-python3-runtime-4.9.3:
      Successfully uninstalled antlr4-python3-runtime-4.9.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [zemberek-python]
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 125.5 MB/s  0:00:00
Looking in links: https://data.p

In [ ]:
# HÜCRE 3: Import ve Zemberek Başlatma

import sys
import typing
# Python 3.13'te kaldırılan typing.io modülünü antlr4 hata vermesin diye yamalıyoruz:
sys.modules['typing.io'] = typing

import torch
from torch_geometric.data import Data
import pandas as pd
from tqdm.auto import tqdm
import pickle # Sözlükleri kaydetmek için
import os

# Zemberek için gerekli importlar
from zemberek import TurkishMorphology, TurkishSentenceNormalizer, TurkishTokenizer

from tokenizers import Tokenizer

# Zemberek'i başlat
# Bu işlem ilk seferde biraz zaman alabilir, model dosyalarını indirir.
morphology = TurkishMorphology.create_with_defaults()

print("Zemberek başarıyla başlatıldı.")

/usr/local/lib/python3.13/dist-packages/zemberek/morphology/turkish_morphology.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
INFO:zemberek.morphology.turkish_morphology:TurkishMorphology instance initialized in 5.317397117614746


2026-10-30 13:19:08,642 - zemberek.morphology.turkish_morphology - INFO
Msg: TurkishMorphology instance initialized in 5.317397117614746

Zemberek başarıyla başlatıldı.


In [ ]:
# HÜCRE 4: Fonksiyon Tanımları
import os
import re
import string
import torch
import pandas as pd
from tqdm.auto import tqdm
import pickle
from torch_geometric.data import Data
import string

def get_morphemes_contextual(words, morphology):
    sentence_morphemes = []

    for word in words:
        word = word.strip()
        if not word or all(char in string.punctuation for char in word):
            sentence_morphemes.append([word.lower()])
            continue

        try:
            word_analysis = morphology.analyze(word)

            if hasattr(word_analysis, 'analysis_results') and word_analysis.analysis_results:
                best = word_analysis.analysis_results[0]
                morphemes_list = []

                # 1. Kök ve POS bilgisini al (Başarıyla test ettiğimiz kısım)
                root = str(best.item.root).lower()
                pos = str(best.item.primary_pos.short_form).lower()
                morphemes_list.append(f"{root}:{pos}")

                # 2. Eklere (Morfemlere) erişim (SENİN SÜRÜMÜNE ÖZEL)
                # Listeden gördüğümüz get_morphemes() metodunu kullanıyoruz
                transitions = best.get_morphemes() if hasattr(best, 'get_morphemes') else []

                for transition in transitions:
                    m_name = ""
                    # Zemberek objesinin içinden morfem adını güvenle çıkarıyoruz
                    if hasattr(transition, 'id_'):
                        m_name = str(transition.id_)
                    elif hasattr(transition, 'name'):
                        m_name = str(transition.name)
                    elif hasattr(transition, 'id'):
                        m_name = str(transition.id)
                    else:
                        m_name = str(transition)

                    # Root ve boş etiketleri tekrar eklememek için filtreliyoruz
                    if m_name and m_name not in ['FreeMorphology', 'Stem', 'None', '']:
                        morphemes_list.append(m_name.lower())

                sentence_morphemes.append(morphemes_list)
            else:
                # Kelime çözülemediyse olduğu gibi bırak
                sentence_morphemes.append([word.lower()])

        except Exception as e:
            # Beklenmedik bir hata olursa çökme, kelimeyi olduğu gibi ekle
            sentence_morphemes.append([word.lower()])

    return sentence_morphemes

def extract_clean_words_from_bpe(sentence, encoded):
    """BPE tokenlarından orijinal kelime sınırlarını kusursuzca çıkarır."""
    bpe_word_ids = encoded.word_ids
    word_bounds = {}

    for i, w_id in enumerate(bpe_word_ids):
        if w_id is not None:
            start, end = encoded.offsets[i]
            if w_id not in word_bounds:
                word_bounds[w_id] = [start, end]
            else:
                word_bounds[w_id][1] = max(word_bounds[w_id][1], end) # BPE parçalarını birleştir

    words_for_zemberek = []
    for w_id in sorted(word_bounds.keys()):
        start, end = word_bounds[w_id]
        # BPE'den gelen baş/son boşlukları SİLDİĞİMİZ kritik nokta!
        clean_word = sentence[start:end].strip()
        words_for_zemberek.append(clean_word)

    return words_for_zemberek

def create_and_save_morph_vocab(data_path, vocab_path, morphology, tokenizer, force_rebuild=False):
    if os.path.exists(vocab_path) and not force_rebuild:
        print(f"Morfem sözlüğü '{vocab_path}' zaten mevcut. Atlanıyor.")
        return

    print("Bağlamsal Morfem Sözlüğü oluşturuluyor...")
    all_morphemes = set()
    df = pd.read_csv(data_path, sep='\t', names=['Turkish', 'English'], on_bad_lines='skip')
    df.dropna(inplace=True)

    for sentence in tqdm(df['Turkish'], desc="Sözlük için analiz yapılıyor"):
        tr_sent = str(sentence)
        encoded = tokenizer.encode(tr_sent)

        # Kelimeleri güvenli bir şekilde çıkarıyoruz
        words_for_zemberek = extract_clean_words_from_bpe(tr_sent, encoded)

        morpheme_data = get_morphemes_contextual(words_for_zemberek, morphology)
        for word_morphemes in morpheme_data:
            all_morphemes.update(word_morphemes)

    morph_vocab = {'<PAD>': 0, '<UNK>': 1}
    for morpheme in sorted(list(all_morphemes)):
        if morpheme not in morph_vocab:
            morph_vocab[morpheme] = len(morph_vocab)

    with open(vocab_path, 'wb') as f:
        pickle.dump(morph_vocab, f)
    print(f"Sözlük kaydedildi: {vocab_path} | Boyut: {len(morph_vocab)}")

def process_and_save_graph_data(input_txt_path, output_pt_path, tokenizer, morph_vocab, morphology):
    if os.path.exists(output_pt_path):
        print(f"'{output_pt_path}' zaten mevcut. Atlanıyor.")
        return

    df = pd.read_csv(input_txt_path, sep='\t', names=['Turkish', 'English'], on_bad_lines='skip')
    df.dropna(inplace=True)

    final_data_list = []

    for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
        tr_sent = str(row['Turkish']).strip()
        en_sent = str(row['English']).strip()

        encoded_tr = tokenizer.encode(tr_sent)
        encoded_en = tokenizer.encode(en_sent)
        bpe_word_ids = encoded_tr.word_ids

        # --- DÜZELTİLEN SATIR BURASI (encoded_tr yapıldı) ---
        words_for_zemberek = extract_clean_words_from_bpe(tr_sent, encoded_tr)

        try:
            morpheme_data_per_word = get_morphemes_contextual(words_for_zemberek, morphology)
        except:
            morpheme_data_per_word = [[w.lower()] for w in words_for_zemberek]

        # Grafın en başına 0. indeks olarak "Boş/PAD" düğümü ekliyoruz.
        node_ids = [morph_vocab.get('<PAD>', 0)]
        word_batch_indices = [0]
        edge_sources, edge_targets = [], []
        node_counter = 1

        for word_idx, morphemes in enumerate(morpheme_data_per_word):
            if word_idx >= len(words_for_zemberek): break

            current_word_batch = word_idx + 1

            for i, morph in enumerate(morphemes):
                node_ids.append(morph_vocab.get(morph, morph_vocab.get('<UNK>', 1)))
                word_batch_indices.append(current_word_batch)

                # İŞTE GRAFI OLUŞTURAN O SİHİRLİ EDGELER BURADA ÇEKİLİYOR:
                if i > 0:
                    edge_sources.append(node_counter - 1)
                    edge_targets.append(node_counter)
                node_counter += 1

        if len(node_ids) == 1: continue # Sadece PAD varsa atla

        src_tensor = torch.tensor(encoded_tr.ids, dtype=torch.long)
        trg_tensor = torch.tensor(encoded_en.ids, dtype=torch.long)

        proc_word_ids = [idx + 1 if idx is not None else 0 for idx in bpe_word_ids]
        src_bpe_word_ids_tensor = torch.tensor(proc_word_ids, dtype=torch.long)

        graph_data = Data(
            x=torch.tensor(node_ids, dtype=torch.long).unsqueeze(1),
            edge_index=torch.tensor([edge_sources, edge_targets], dtype=torch.long),
            word_batch=torch.tensor(word_batch_indices, dtype=torch.long)
        )

        final_data_list.append({
            'src_bpe': src_tensor,
            'graph_data': graph_data,
            'trg_bpe': trg_tensor,
            'src_bpe_word_ids': src_bpe_word_ids_tensor,
            'tokens': encoded_tr.tokens
        })

    torch.save(final_data_list, output_pt_path)
    print(f"Saved {len(final_data_list)} examples to {output_pt_path}")

In [ ]:
# HÜCRE 5: EN2TR ham veriyi train/valid/test olarak böl
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Ham EN2TR dosyasının yolu (Drive'a yüklemiş olman gerekiyor)
EN2TR_RAW_PATH = '/content/drive/MyDrive/Tez_Veri/EN2TR/TR2EN.txt'

# Çıktı klasörü
OUTPUT_DIR = '/content/drive/MyDrive/Tez_Veri/EN2TR'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Format: English<TAB>Turkish
df = pd.read_csv(
    EN2TR_RAW_PATH,
    sep='\t',
    names=['English', 'Turkish'],
    on_bad_lines='skip'
)
df.dropna(inplace=True)
df = df.drop_duplicates()
print(f"Toplam (duplicate temizleme sonrası): {len(df)} cümle çifti")

# Notebook'taki preprocessing kodu ['Turkish', 'English'] sırasını bekliyor
# Sütun sırasını çevir:
df = df[['Turkish', 'English']]

# WMT18 ile simetrik boyutta valid/test:
# Toplam ~473K → train ~467K, valid 3K, test 3K
train_df, temp_df  = train_test_split(df, test_size=6000, random_state=42)
valid_df, test_df  = train_test_split(temp_df, test_size=3000, random_state=42)

print(f"Train: {len(train_df)}")
print(f"Valid: {len(valid_df)}")
print(f"Test : {len(test_df)}")

# TSV olarak kaydet (preprocessing kodunun beklediği format)
train_df.to_csv(f'{OUTPUT_DIR}/train.tsv', sep='\t', header=False, index=False)
valid_df.to_csv(f'{OUTPUT_DIR}/val.tsv',   sep='\t', header=False, index=False)
test_df.to_csv(f'{OUTPUT_DIR}/test.tsv',   sep='\t', header=False, index=False)

print(f"\n✅ Bölme tamamlandı. Dosyalar: {OUTPUT_DIR}/")

Toplam (duplicate temizleme sonrası): 473034 cümle çifti
Train: 467034
Valid: 3000
Test : 3000

✅ Bölme tamamlandı. Dosyalar: /content/drive/MyDrive/Tez_Veri/EN2TR/


In [ ]:
# HÜCRE 6: EN2TR için yeni BPE tokenizer eğit
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import pandas as pd

VOCAB_SIZE = 40000   # WMT18 ile aynı boyut — fair karşılaştırma için kritik

tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<PAD>", "<UNK>", "<SOS>", "<EOS>"],
    show_progress=True
)

# Train verisinden hem TR hem EN cümleleri tokenize için kullan
def get_training_corpus(tsv_path):
    df = pd.read_csv(
        tsv_path, sep='\t',
        names=['Turkish', 'English'],
        on_bad_lines='skip'
    )
    df.dropna(inplace=True)
    for _, row in df.iterrows():
        yield str(row['Turkish'])
        yield str(row['English'])

TOKENIZER_OUT = '/content/drive/MyDrive/Tez_Veri/EN2TR/bpe_tokenizer_en2tr.json'
TRAIN_TSV     = '/content/drive/MyDrive/Tez_Veri/EN2TR/train.tsv'

tokenizer.train_from_iterator(get_training_corpus(TRAIN_TSV), trainer=trainer)
tokenizer.save(TOKENIZER_OUT)

# Doğrulama
loaded = Tokenizer.from_file(TOKENIZER_OUT)
print(f"\n✅ EN2TR BPE tokenizer kaydedildi: {TOKENIZER_OUT}")
print(f"   Boyut: {loaded.get_vocab_size()}")

# Özel tokenları kontrol et
for tok in ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]:
    print(f"   {tok}: ID={loaded.token_to_id(tok)}")


✅ EN2TR BPE tokenizer kaydedildi: /content/drive/MyDrive/Tez_Veri/EN2TR/bpe_tokenizer_en2tr.json
   Boyut: 40000
   <PAD>: ID=0
   <UNK>: ID=1
   <SOS>: ID=2
   <EOS>: ID=3


In [ ]:
import pandas as pd
D='/content/drive/MyDrive/Tez_Veri/EN2TR'
tr=pd.read_csv(f'{D}/train.tsv',sep='\t',names=['tr','en'],on_bad_lines='skip').dropna()
bpe_len  = tr['tr'].apply(lambda s: len(tokenizer.encode(str(s)).ids))
word_len = tr['tr'].apply(lambda s: len(str(s).split()))
print("Max BPE token:", bpe_len.max(),  "| >100 olan cümle:", int((bpe_len>100).sum()))
print("Max kelime   :", word_len.max(), "| >100 olan cümle:", int((word_len>100).sum()))

Max BPE token: 46 | >100 olan cümle: 0
Max kelime   : 36 | >100 olan cümle: 0


In [ ]:
# HÜCRE 7 (EN2TR): Ana iş akışı (morph vocab + graph data)
import os
import pickle
from tokenizers import Tokenizer
from zemberek import TurkishMorphology

morphology = TurkishMorphology.create_with_defaults()

# === EN2TR YOLLARI ===
EN2TR_DIR        = '/content/drive/MyDrive/Tez_Veri/EN2TR'
TRAIN_INPUT_PATH = f'{EN2TR_DIR}/train.tsv'
VALID_INPUT_PATH = f'{EN2TR_DIR}/val.tsv'
TEST_INPUT_PATH  = f'{EN2TR_DIR}/test.tsv'
TOKENIZER_FILE   = f'{EN2TR_DIR}/bpe_tokenizer_en2tr.json'
MORPH_VOCAB_PATH = f'{EN2TR_DIR}/morph_vocab_en2tr.pkl'

# Çıktı dosyaları (EN2TR'ye özel isimle)
TRAIN_OUT = f'{EN2TR_DIR}/processed_train_graph_data.pt'
VALID_OUT = f'{EN2TR_DIR}/processed_valid_graph_data.pt'
TEST_OUT  = f'{EN2TR_DIR}/processed_test_graph_data.pt'

# 1. Tokenizer'ı yükle
try:
    tokenizer = Tokenizer.from_file(TOKENIZER_FILE)
    print(f"✅ EN2TR BPE tokenizer yüklendi. Boyut: {tokenizer.get_vocab_size()}")
except Exception as e:
    print(f"❌ HATA: Tokenizer yüklenemedi. BPE eğitim hücresini çalıştırdın mı? Hata: {e}")

# 2. Morfem Sözlüğü Oluştur (sadece train verisinden)
create_and_save_morph_vocab(
    TRAIN_INPUT_PATH, MORPH_VOCAB_PATH,
    morphology, tokenizer, force_rebuild=True
)

# 3. Sözlüğü Yükle
with open(MORPH_VOCAB_PATH, 'rb') as f:
    morph_vocab = pickle.load(f)

# 4. Veri Setlerini İşle
print("\n--- EN2TR Veri Setleri İşleniyor ---")
process_and_save_graph_data(TRAIN_INPUT_PATH, TRAIN_OUT, tokenizer, morph_vocab, morphology)
process_and_save_graph_data(VALID_INPUT_PATH, VALID_OUT, tokenizer, morph_vocab, morphology)
process_and_save_graph_data(TEST_INPUT_PATH,  TEST_OUT,  tokenizer, morph_vocab, morphology)

print("\n🚀 EN2TR ÖN İŞLEME TAMAMLANDI!")

INFO:zemberek.morphology.turkish_morphology:TurkishMorphology instance initialized in 5.828374624252319


2026-10-28 13:56:17,481 - zemberek.morphology.turkish_morphology - INFO
Msg: TurkishMorphology instance initialized in 5.828374624252319

✅ EN2TR BPE tokenizer yüklendi. Boyut: 40000
Bağlamsal Morfem Sözlüğü oluşturuluyor...


Sözlük için analiz yapılıyor:   0%|          | 0/467034 [00:00<?, ?it/s]

Sözlük kaydedildi: /content/drive/MyDrive/Tez_Veri/EN2TR/morph_vocab_en2tr.pkl | Boyut: 14421

--- EN2TR Veri Setleri İşleniyor ---
'/content/drive/MyDrive/Tez_Veri/EN2TR/processed_train_graph_data.pt' zaten mevcut. Atlanıyor.
'/content/drive/MyDrive/Tez_Veri/EN2TR/processed_valid_graph_data.pt' zaten mevcut. Atlanıyor.
'/content/drive/MyDrive/Tez_Veri/EN2TR/processed_test_graph_data.pt' zaten mevcut. Atlanıyor.

🚀 EN2TR ÖN İŞLEME TAMAMLANDI!


In [ ]:
# ADIM 2 — YENİ HÜCRE: Zemberek belirsizlik / çözümleme-hatası / UNK oranı
import pandas as pd, string
from tqdm.auto import tqdm

print("HÜCRE B:")


TEST_TSV = '/content/drive/MyDrive/Tez_Veri/EN2TR/test.tsv'
df = pd.read_csv(TEST_TSV, sep='\t', names=['Turkish','English'], on_bad_lines='skip').dropna()

n_words = n_ambig = n_fail = n_unk = 0
unk_id = morph_vocab.get('<UNK>', 1)

for sent in tqdm(df['Turkish'], desc="Zemberek denetimi"):
    enc = tokenizer.encode(str(sent))
    words = extract_clean_words_from_bpe(str(sent), enc)
    for w in words:
        w = w.strip()
        if not w or all(c in string.punctuation for c in w):
            continue
        n_words += 1
        a = morphology.analyze(w)
        res = a.analysis_results if hasattr(a, 'analysis_results') else []
        if not res:
            n_fail += 1                       # çözülemeyen kelime (fallback)
        elif len(res) > 1:
            n_ambig += 1                      # birden fazla çözüm = belirsiz
        # birimler sözlükte var mı?
        morphs = get_morphemes_contextual([w], morphology)[0]
        if any(morph_vocab.get(m, unk_id) == unk_id for m in morphs):
            n_unk += 1

print(f"Toplam içerik kelimesi   : {n_words}")
print(f"Belirsiz (>1 çözüm)      : {n_ambig}  (%{100*n_ambig/n_words:.1f})")
print(f"Çözülemeyen (fallback)   : {n_fail}  (%{100*n_fail/n_words:.1f})")
print(f"≥1 <UNK> birim içeren    : {n_unk}  (%{100*n_unk/n_words:.1f})")

Zemberek denetimi:   0%|          | 0/3000 [00:00<?, ?it/s]

Toplam içerik kelimesi   : 14850
Belirsiz (>1 çözüm)      : 8872  (%59.7)
Çözülemeyen (fallback)   : 99  (%0.7)
≥1 <UNK> birim içeren    : 24  (%0.2)


In [ ]:
# ADIM 2 — YENİ HÜCRE: dedup sayıları + bölünmeler arası örtüşme denetimi

print("HÜCRE E:")
import pandas as pd
D = '/content/drive/MyDrive/Tez_Veri/EN2TR'
rd = lambda f: pd.read_csv(f, sep='\t', names=['tr','en'], on_bad_lines='skip').dropna()
tr_, va, te = rd(f'{D}/train.tsv'), rd(f'{D}/val.tsv'), rd(f'{D}/test.tsv')

norm = lambda s: ' '.join(str(s).lower().split())
for name, d in [('train',tr_),('val',va),('test',te)]:
    pair_dups = len(d) - len(d.drop_duplicates())
    src_dups  = d['tr'].map(norm).duplicated().sum()
    print(f"{name:5} | satır {len(d):>6} | çift-tekrar {pair_dups} | kaynak-tekrar {src_dups}")

tr_src, va_src, te_src = set(tr_['tr'].map(norm)), set(va['tr'].map(norm)), set(te['tr'].map(norm))
print(f"\nKaynak örtüşmesi  train∩test: {len(tr_src & te_src)}")
print(f"Kaynak örtüşmesi  train∩val : {len(tr_src & va_src)}")
print(f"Benzersiz test kaynak cümlesi: {len(te_src)} / {len(te)}")

train | satır 467034 | çift-tekrar 0 | kaynak-tekrar 30025
val   | satır   3000 | çift-tekrar 0 | kaynak-tekrar 1
test  | satır   3000 | çift-tekrar 0 | kaynak-tekrar 3

Kaynak örtüşmesi  train∩test: 333
Kaynak örtüşmesi  train∩val : 341
Benzersiz test kaynak cümlesi: 2997 / 3000


In [ ]:
# HÜCRE 8: (SANITY CHECK)
import os
import torch
import pickle

print("==========================================================")
print("🧐 KAPSAMLI VERİ MANTIĞI VE HİZALAMA KONTROLÜ BAŞLIYOR")
print("==========================================================\n")

# 1. SÖZLÜK KONTROLÜ
print("--- 1. MORFEM SÖZLÜĞÜ KONTROLÜ ---")
try:
    with open('/content/drive/MyDrive/Tez_Veri/EN2TR/morph_vocab_en2tr.pkl', 'rb') as f:
        mv = pickle.load(f)
    print(f"✅ Sözlük başarıyla yüklendi. Toplam Morfem/Kök/Ek sayısı: {len(mv)}")
    print(f"✅ Özel Tokenlar: <PAD> -> {mv.get('<PAD>')}, <UNK> -> {mv.get('<UNK>')}")
    if len(mv) < 100:
        print("⚠️ UYARI: Sözlük boyutu çok küçük! Beklenen binlerce olmalıydı.")
except Exception as e:
    print(f"❌ Sözlük yüklenirken hata: {e}")

print("\n--- 2. VERİ SETİ BOYUTLARI KONTROLÜ ---")
try:
    train_data = torch.load('/content/drive/MyDrive/Tez_Veri/EN2TR/processed_train_graph_data.pt', weights_only=False)
    valid_data = torch.load('/content/drive/MyDrive/Tez_Veri/EN2TR/processed_valid_graph_data.pt', weights_only=False)
    test_data  = torch.load('/content/drive/MyDrive/Tez_Veri/EN2TR/processed_test_graph_data.pt',  weights_only=False)
    print(f"✅ Eğitim (Train) Seti: {len(train_data)} örnek")
    print(f"✅ Doğrulama (Valid) Seti: {len(valid_data)} örnek")
    print(f"✅ Sınav (Test) Seti: {len(test_data)} örnek")
except Exception as e:
    print(f"❌ Veri setleri yüklenirken hata: {e}")

print("\n--- 3. DETAYLI ÖRNEK İNCELEMESİ (Train[0]) ---")
try:
    sample = train_data[0] # İlk cümleyi mikroskop altına alıyoruz

    src_bpe = sample['src_bpe']
    trg_bpe = sample['trg_bpe']
    src_bpe_word_ids = sample['src_bpe_word_ids']
    graph = sample['graph_data']
    tokens = sample['tokens']

    # Ters sözlük (ID -> Metin)
    inv_mv = {v: k for k, v in mv.items()}
    node_texts = [inv_mv.get(idx.item(), '<UNK>') for idx in graph.x.flatten()]

    # A. Tensör Şekilleri (Shapes)
    print("🔹 A. PyTorch Tensör Boyutları:")
    print(f"   - Kaynak BPE (TR) Boyutu: {src_bpe.shape}")
    print(f"   - Hedef BPE (EN) Boyutu: {trg_bpe.shape} (Modelin üreteceği İngilizce kelimeler)")
    print(f"   - Word ID Hizalama Boyutu: {src_bpe_word_ids.shape} (Kaynak BPE ile aynı olmalı)")
    print(f"   - Graf Düğüm (Node) Boyutu: {graph.x.shape} (N, 1 formunda olmalı)")
    print(f"   - Graf Bağlantı (Edge) Boyutu: {graph.edge_index.shape} (2, E formunda olmalı)")
    print(f"   - Word Batch Boyutu: {graph.word_batch.shape} (Graf düğümleriyle aynı uzunlukta olmalı)")

    assert src_bpe.shape == src_bpe_word_ids.shape, "HATA: Token sayısı ile Word ID sayısı uyuşmuyor!"
    assert graph.x.size(0) == graph.word_batch.size(0), "HATA: Düğüm sayısı ile Word Batch sayısı uyuşmuyor!"

    # B. Graf Yapısı
    print("\n🔹 B. Zemberek Graf Yapısı:")
    print(f"   - Toplam Düğüm Sayısı: {graph.x.size(0)}")
    print(f"   - Toplam Bağlantı (Edge) Sayısı: {graph.edge_index.size(1)}")
    if graph.edge_index.size(1) > 0:
        print(f"   - İlk 3 Bağlantı (Kim kime bağlanıyor): \n{graph.edge_index[:, :3]}")

    # C. Görsel Kelime Hizalama Haritası
    print("\n🔹 C. GÖRSEL KELİME-GRAF HİZALAMA HARİTASI:")
    print(f"{'Word ID':<8} | {'BPE Alt-Kelimeleri (Tokens)':<35} | {'Zemberek Graf Düğümleri (Nodes)'}")
    print("-" * 90)

    # ID'lere göre gruplama
    from collections import defaultdict
    id_to_tokens = defaultdict(list)
    for tok, wid in zip(tokens, src_bpe_word_ids.tolist()):
        id_to_tokens[wid].append(tok)

    id_to_nodes = defaultdict(list)
    for node, wid in zip(node_texts, graph.word_batch.tolist()):
        id_to_nodes[wid].append(node)

    # Ekrana haritayı bas
    all_wids = sorted(set(src_bpe_word_ids.tolist() + graph.word_batch.tolist()))
    for wid in all_wids:
        tok_str = " + ".join(id_to_tokens[wid]) if wid in id_to_tokens else "---"
        node_str = " -> ".join(id_to_nodes[wid]) if wid in id_to_nodes else "---"
        print(f"ID: {wid:<4} | {tok_str:<35} | {node_str}")

    print("\n✅ KONTROL TAMAMLANDI: Eğer tensör boyutlarında bir Assert hatası almadıysan ve yukarıdaki haritada BPE parçaları ile Morfem kökleri aynı Word ID satırında mantıklı bir şekilde hizalanmışsa, VERİN %100 KUSURSUZDUR!")
    print("Artık HÜCRE 7'yi çalıştırarak bu değerli '.pt' dosyalarını indirebilirsin.")

except Exception as e:
    print(f"\n❌ KONTROL SIRASINDA KRİTİK HATA OLUŞTU: {e}")

🧐 KAPSAMLI VERİ MANTIĞI VE HİZALAMA KONTROLÜ BAŞLIYOR

--- 1. MORFEM SÖZLÜĞÜ KONTROLÜ ---
✅ Sözlük başarıyla yüklendi. Toplam Morfem/Kök/Ek sayısı: 14428
✅ Özel Tokenlar: <PAD> -> 0, <UNK> -> 1

--- 2. VERİ SETİ BOYUTLARI KONTROLÜ ---
✅ Eğitim (Train) Seti: 467034 örnek
✅ Doğrulama (Valid) Seti: 3000 örnek
✅ Sınav (Test) Seti: 3000 örnek

--- 3. DETAYLI ÖRNEK İNCELEMESİ (Train[0]) ---
🔹 A. PyTorch Tensör Boyutları:
   - Kaynak BPE (TR) Boyutu: torch.Size([4])
   - Hedef BPE (EN) Boyutu: torch.Size([5]) (Modelin üreteceği İngilizce kelimeler)
   - Word ID Hizalama Boyutu: torch.Size([4]) (Kaynak BPE ile aynı olmalı)
   - Graf Düğüm (Node) Boyutu: torch.Size([13, 1]) (N, 1 formunda olmalı)
   - Graf Bağlantı (Edge) Boyutu: torch.Size([2, 9]) (2, E formunda olmalı)
   - Word Batch Boyutu: torch.Size([13]) (Graf düğümleriyle aynı uzunlukta olmalı)

🔹 B. Zemberek Graf Yapısı:
   - Toplam Düğüm Sayısı: 13
   - Toplam Bağlantı (Edge) Sayısı: 9
   - İlk 3 Bağlantı (Kim kime bağlanıyor): 
tenso

BURADA BİTİYOR

In [ ]:
# HÜCRE 7: Dosyaları İndir
from google.colab import files
import os

# İsimlerin HÜCRE 5 ile birebir aynı olduğundan emin olalım
# Eğer HÜCRE 5'te değişken kullanmadıysan burada manuel yazabilirsin:
files_to_download = [
    'morph_vocab.pkl',
    'processed_train_graph_data.pt',
    'processed_valid_graph_data.pt',
    'processed_test_graph_data.pt',
    'bpe_tokenizer.json' # Makale için bunu da yedekle!
]

print("\n--- Oluşturulan dosyalar hazırlanıyor ve indiriliyor... ---")

for file_path in files_to_download:
    if os.path.exists(file_path):
        print(f"'{file_path}' indiriliyor...")
        files.download(file_path)
    else:
        print(f"⚠️ UYARI: '{file_path}' bulunamadı. Lütfen HÜCRE 5'te dosyanın bu isimle kaydedildiğinden emin olun.")

print("\nİndirme işlemi başlatıldı. Lütfen tarayıcınızın çoklu dosya indirme iznini onaylayın.")


--- Oluşturulan dosyalar hazırlanıyor ve indiriliyor... ---
'morph_vocab.pkl' indiriliyor...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'processed_train_graph_data.pt' indiriliyor...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (6 outputs left out)

'bpe_tokenizer.json' indiriliyor...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


İndirme işlemi başlatıldı. Lütfen tarayıcınızın çoklu dosya indirme iznini onaylayın.


In [ ]:
analysis = morphology.analyze("özelleştirme")
print(f"Obje Tipi: {type(analysis)}")
print("\n--- WORDANALYSIS OBJESININ İÇİNDEKİLER ---")
print([m for m in dir(analysis) if not m.startswith('_')])

Obje Tipi: <class 'zemberek.morphology.analysis.word_analysis.WordAnalysis'>

--- WORDANALYSIS OBJESININ İÇİNDEKİLER ---
['EMPTY_INPUT_RESULT', 'analysis_results', 'inp', 'is_correct', 'normalized_input']


In [ ]:
# 1. Zemberek analizini yap
word_analysis = morphology.analyze("özelleştirme")

# 2. İlk sonucu al
best = word_analysis.analysis_results[0]

print(f"İç Obje Tipi: {type(best)}")
print("\n--- İÇ OBJENİN ÖZELLİKLERİ VE METODLARI ---")
print([m for m in dir(best) if not m.startswith('_')])

# 3. Kök ve eklere erişmeyi deneyelim (Büyük ihtimalle burada patlayacak ve asıl hatayı göreceğiz)
print("\nKök test ediliyor...")
item = best.item if hasattr(best, 'item') else best.get_item()
print(item.root)

İç Obje Tipi: <class 'zemberek.morphology.analysis.single_analysis.SingleAnalysis'>

--- İÇ OBJENİN ÖZELLİKLERİ VE METODLARI ---
['MorphemeData', 'MorphemeGroup', 'contains_informal_morpheme', 'contains_morpheme', 'copy_for', 'dummy', 'empty_morpheme_cache', 'format_morpheme_string', 'format_string', 'from_search_path', 'get_ending', 'get_group', 'get_morphemes', 'get_stem', 'group_boundaries', 'hash_', 'is_runtime', 'is_unknown', 'item', 'morpheme_data_list', 'surface_form', 'unknown']

Kök test ediliyor...
özel
